# Listener Prior — Retrieval Model + Shared Index (3 datasets, Option A)

Trains a **retrieval** model (SentenceTransformers) to predict **keyterms** and **keywords** for the **next utterance** (Option A, role-agnostic).
Selects the **best checkpoint by Recall@20**, **separately for keyterms and keywords**, and saves:

- `best_model/` (encoder)
- `shared_index/` (FAISS index + candidates + metadata)
- `best_metrics.json`, `performance.json`

Also prints example conversations with predictions.

## Datasets
- Public dialog datasets (repo loader or HF fallback)
- `examples/multi_restaurant_phone_orders_2000.csv`


In [ ]:
# ---------------------------
# CONFIG (edit as needed)
# ---------------------------
REPO_URL = "https://github.com/ebilal/fSTT.git"
PROJECT_DIR = "/content/listener-prior"

BASE_EMBEDDER = "sentence-transformers/paraphrase-MiniLM-L3-v2"  # pretrained; we fine-tune
HISTORY_TURNS = 8
MAX_KEYWORDS = 30
MAX_KEYTERMS = 30

# Training
EPOCHS = 6
BATCH_SIZE = 256
LR = 2e-5
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS = 1e-8

# Selection metric
TOPK = 20
# We select best by KEYTERMS Recall@20 (and still log keywords Recall@20)
SELECTION_FIELD = "val_recall@20_keyterms"
VAL_EXAMPLES_FOR_FAST_EVAL = None  # e.g. 5000

# Saving (Google Drive)
RUN_NAME = "retrieval_minilm_l3_optionA_3datasets_v2_sep_metrics"
DRIVE_ROOT = "/content/drive/MyDrive/listener_prior_runs"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/best_model"
INDEX_DIR = f"{RUN_DIR}/shared_index"

RESTAURANT_CSV = "examples/multi_restaurant_phone_orders_5000.csv"


## Mount Google Drive + set HF_TOKEN from Colab Secrets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found. Public datasets/models should still work without it.')


## Clone repo + install dependencies


In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt
!python -m pip install faiss-cpu accelerate

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())


## Load conversations (public + restaurant CSV)


In [ ]:
import os, json
import pandas as pd
from typing import List, Dict, Any

def load_public_conversations() -> List[Dict[str, Any]]:
    convos: List[Dict[str, Any]] = []
    try:
        from src import data as data_mod
        for fn_name in ["load_dual_dataset_conversations", "load_public_conversations", "get_dual_dataset"]:
            if hasattr(data_mod, fn_name):
                out = getattr(data_mod, fn_name)()
                if isinstance(out, dict):
                    for _, v in out.items():
                        if isinstance(v, list):
                            convos.extend(v)
                elif isinstance(out, list):
                    convos = out
                if convos:
                    print(f"Loaded public via src.data.{fn_name}: {len(convos)}")
                    return convos
    except Exception as e:
        print("Repo loader not available / failed:", repr(e))

    import datasets

    def _normalize_role(role: str) -> str:
        role = (role or "").strip().lower()
        if role in {"user", "customer", "human", "client", "guest"}:
            return "USER"
        if role in {"system", "assistant", "agent", "bot", "server"}:
            return "SYSTEM"
        return "SYSTEM"

    def _extract_text(turn: Dict) -> str:
        for key in ["text", "utterance", "transcript", "content", "sentence"]:
            if key in turn and isinstance(turn[key], str):
                return turn[key]
        return ""

    def _extract_turns_from_item(item: Dict) -> List[Dict[str, str]]:
        turns: List[Dict[str, str]] = []

        # DailyDialog (roskoN/dailydialog) style: a single list of utterances.
        # Roles are not explicitly provided; alternate USER/SYSTEM by index.
        if "utterances" in item and isinstance(item["utterances"], list) and item["utterances"]:
            utterances = item["utterances"]
            for i, text in enumerate(utterances):
                if not isinstance(text, str):
                    continue
                text = text.strip()
                if not text:
                    continue
                role = "USER" if i % 2 == 0 else "SYSTEM"
                turns.append({"speaker": role, "text": text})
            if turns:
                return turns

        if "turns" in item and isinstance(item["turns"], dict):
            speakers = item["turns"].get("speaker") or []
            utterances = item["turns"].get("utterance") or []
            for speaker, text in zip(speakers, utterances):
                role = "USER" if str(speaker) == "0" else "SYSTEM"
                text = (text or "").strip()
                if text:
                    turns.append({"speaker": role, "text": text})
            return turns

        turns_list = None
        for key in ["turns", "dialogue", "dialog", "utterances", "messages"]:
            if key in item and isinstance(item[key], list):
                turns_list = item[key]
                break
        if turns_list is None:
            return []

        for idx, turn in enumerate(turns_list):
            if isinstance(turn, str):
                role = "USER" if idx % 2 == 0 else "SYSTEM"
                text = turn
            elif isinstance(turn, dict):
                role = _normalize_role(turn.get("speaker") or turn.get("role") or turn.get("participant") or "")
                text = _extract_text(turn)
                if not text and "utterances" in turn and isinstance(turn["utterances"], str):
                    text = turn["utterances"]
            else:
                continue
            text = (text or "").strip()
            if text:
                turns.append({"speaker": role, "text": text})
        return turns

    # Prefer the HF-hosted DailyDialog mirror (no external zip link), with legacy fallback.
    DATASET_SOURCES = [
        ("dailydialog", ["roskoN/dailydialog", "daily_dialog"]),
        ("multiwoz", ["pfb30/multi_woz_v22"]),
    ]

    token = (
        os.environ.get("HF_TOKEN")
        or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        or os.environ.get("HUGGING_FACE_HUB_TOKEN")
        or None
    )

    def _load(repo_id: str):
        if token:
            try:
                return datasets.load_dataset(repo_id, split="train", token=token, trust_remote_code=True)
            except TypeError:
                return datasets.load_dataset(repo_id, split="train", use_auth_token=token, trust_remote_code=True)
        return datasets.load_dataset(repo_id, split="train", trust_remote_code=True)

    print("Using HF fallback datasets:", DATASET_SOURCES)

    for ds_name, repo_ids in DATASET_SOURCES:
        dataset = None
        used_repo = None
        for repo_id in repo_ids:
            try:
                dataset = _load(repo_id)
                used_repo = repo_id
                break
            except Exception as e:
                print(f"Skipping {repo_id} due to load error: {repr(e)}")
                continue
        if dataset is None:
            continue
        if ds_name == "dailydialog" and used_repo == "daily_dialog":
            print("Warning: using legacy 'daily_dialog' source; prefer 'roskoN/dailydialog'.")

        for i, item in enumerate(dataset):
            turns = _extract_turns_from_item(item)
            if not turns:
                dialog = item.get("dialog")
                if isinstance(dialog, list):
                    turns = [{"speaker": "USER" if j % 2 == 0 else "SYSTEM", "text": str(t)} for j, t in enumerate(dialog) if isinstance(t, str)]
            if len(turns) >= 2:
                convos.append({"dialog_id": f"{used_repo}:train:{i}", "turns": turns})

    print("Loaded HF public conversations:", len(convos))
    return convos

def load_restaurant_csv(path: str) -> List[Dict[str, Any]]:
    assert os.path.exists(path), f"Missing CSV at {path}."
    df = pd.read_csv(path)
    required = {"dialog_id", "utterance_id", "speaker", "text"}
    assert required.issubset(set(df.columns)), f"CSV must include {required}, got {set(df.columns)}"
    convos: List[Dict[str, Any]] = []
    for did, g in df.sort_values(["dialog_id", "utterance_id"]).groupby("dialog_id"):
        turns = [
            {"speaker": _normalize_role(str(r["speaker"])), "text": str(r["text"])}
            for _, r in g.iterrows()
        ]
        if len(turns) >= 2:
            convos.append({"dialog_id": f"restaurant:{did}", "turns": turns})
    print("Loaded restaurant conversations:", len(convos))
    return convos

public_convos = load_public_conversations()
restaurant_convos = load_restaurant_csv(RESTAURANT_CSV)
all_convos = public_convos + restaurant_convos
print("TOTAL conversations:", len(all_convos))


In [ ]:
# Sanity check: show a few dialogs from each source
from itertools import islice

def _print_samples(convos, label, n=2):
    print("
" + "="*80)
    print(f"{label} (showing {n})")
    print("="*80)
    for c in list(islice(convos, n)):
        print("dialog_id:", c["dialog_id"])
        for t in c["turns"][:3]:
            print(f"  {t['speaker']}: {t['text']}")
        if len(c["turns"]) > 3:
            print("  ...")

public_only = [c for c in all_convos if not c["dialog_id"].startswith("restaurant:")]
restaurant_only = [c for c in all_convos if c["dialog_id"].startswith("restaurant:")]

_print_samples(public_only, "Public datasets")
_print_samples(restaurant_only, "Restaurant CSV")


## Build Option A examples + global candidate pool


In [ ]:
from src.prior import extract_priors

def build_examples(convos, history_turns: int):
    exs = []
    for c in convos:
        turns = c["turns"]
        for t in range(1, len(turns)):
            hist = turns[max(0, t-history_turns):t]
            target = turns[t]
            inp = "\n".join([f'{h["speaker"]}: {h["text"]}' for h in hist]).strip()

            pri = extract_priors(target["text"], max_keywords=MAX_KEYWORDS, max_keyterms=MAX_KEYTERMS)
            tgt = "keyterms: " + "; ".join(pri.get("keyterms", [])) + "\n"
            tgt += "keywords: " + "; ".join(pri.get("keywords", []))

            exs.append({"dialog_id": c["dialog_id"], "turn_idx": t, "input_text": inp, "target_text": tgt})
    return exs

examples = build_examples(all_convos, HISTORY_TURNS)
candidates = sorted(set(e["target_text"] for e in examples if e["target_text"]))
print("examples:", len(examples), "unique candidates:", len(candidates))


## Split train/val/test by dialog id


In [ ]:
from collections import defaultdict
import random

rng = random.Random(7)
by_dialog = defaultdict(list)
for e in examples:
    by_dialog[e["dialog_id"]].append(e)

dialog_ids = list(by_dialog.keys())
rng.shuffle(dialog_ids)

n = len(dialog_ids)
n_train = int(0.8*n)
n_val = int(0.1*n)

train_ids = set(dialog_ids[:n_train])
val_ids   = set(dialog_ids[n_train:n_train+n_val])
test_ids  = set(dialog_ids[n_train+n_val:])

train_ex = [e for did in train_ids for e in by_dialog[did]]
val_ex   = [e for did in val_ids for e in by_dialog[did]]
test_ex  = [e for did in test_ids for e in by_dialog[did]]

print("dialogs:", n, "train/val/test:", len(train_ids), len(val_ids), len(test_ids))
print("examples:", len(train_ex), len(val_ex), len(test_ex))


## Recall@20 metrics split: keyterms vs keywords


In [ ]:
def parse_keyterms_keywords(text: str):
    keyterms = []
    keywords = []
    for line in text.splitlines():
        line = line.strip()
        if line.lower().startswith("keyterms:"):
            rhs = line.split(":",1)[1]
            keyterms = [t.strip() for t in rhs.split(";") if t.strip()]
        elif line.lower().startswith("keywords:"):
            rhs = line.split(":",1)[1]
            keywords = [t.strip() for t in rhs.split(";") if t.strip()]
    # de-dup preserving order
    def dedup(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out
    return dedup(keyterms), dedup(keywords)

def recall_at_k(gt_list, pred_list, k):
    gt_set = set(gt_list)
    if not gt_set:
        return 0.0
    topk = pred_list[:k]
    return len(set(topk) & gt_set) / len(gt_set)

def recall20_from_retrieved(gt_text: str, retrieved_texts: list[str], k=20):
    gt_terms, gt_words = parse_keyterms_keywords(gt_text)

    pred_terms=[]
    pred_words=[]
    for cand in retrieved_texts:
        ct, cw = parse_keyterms_keywords(cand)
        pred_terms.extend(ct)
        pred_words.extend(cw)

    # unique preserving order
    def uniq(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out

    pred_terms = uniq(pred_terms)
    pred_words = uniq(pred_words)

    return {
        "recall@20_keyterms": recall_at_k(gt_terms, pred_terms, k),
        "recall@20_keywords": recall_at_k(gt_words, pred_words, k),
    }


## Train retrieval model + select best by VAL Recall@20 (keyterms)


In [ ]:
import os, json
import numpy as np
import faiss
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

os.makedirs(RUN_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(BASE_EMBEDDER, device=device)
loss = losses.MultipleNegativesRankingLoss(model)

train_pairs = [InputExample(texts=[e["input_text"], e["target_text"]]) for e in train_ex]
train_dl = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

def build_faiss_index(m, cand_texts):
    emb = m.encode(cand_texts, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    dim = emb.shape[1]
    idx = faiss.IndexFlatIP(dim)
    idx.add(emb.astype(np.float32))
    return idx

@torch.no_grad()
def eval_metrics(m, cand_texts, eval_ex, limit=None):
    if limit is not None and limit < len(eval_ex):
        eval_ex = eval_ex[:limit]
    idx = build_faiss_index(m, cand_texts)
    bs = 512
    sum_terms=0.0; sum_words=0.0
    for i in range(0, len(eval_ex), bs):
        chunk = eval_ex[i:i+bs]
        queries = [e["input_text"] for e in chunk]
        q_emb = m.encode(queries, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        D, I = idx.search(q_emb.astype(np.float32), TOPK)
        for row, ex in enumerate(chunk):
            retrieved = [cand_texts[j] for j in I[row] if j >= 0]
            m20 = recall20_from_retrieved(ex["target_text"], retrieved, k=TOPK)
            sum_terms += m20["recall@20_keyterms"]
            sum_words += m20["recall@20_keywords"]
    n = max(1, len(eval_ex))
    return {"recall@20_keyterms": float(sum_terms/n), "recall@20_keywords": float(sum_words/n)}

best = {SELECTION_FIELD: -1.0, "epoch": None}
history = []

steps_per_epoch = len(train_dl)
warmup_steps = int(WARMUP_RATIO * steps_per_epoch)

for epoch in range(1, EPOCHS+1):
    print(f"\n===== EPOCH {epoch}/{EPOCHS} =====")
    model.fit(
        train_objectives=[(train_dl, loss)],
        epochs=1,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": LR, "weight_decay": WEIGHT_DECAY, "betas": (ADAM_BETA1, ADAM_BETA2), "eps": ADAM_EPS},
        show_progress_bar=True,
    )
    val_m = eval_metrics(model, candidates, val_ex, limit=VAL_EXAMPLES_FOR_FAST_EVAL)
    print(f"VAL Recall@20 keyterms: {val_m['recall@20_keyterms']:.6f} | keywords: {val_m['recall@20_keywords']:.6f}")
    row = {"epoch": epoch, "val_recall@20_keyterms": val_m["recall@20_keyterms"], "val_recall@20_keywords": val_m["recall@20_keywords"]}
    history.append(row)

    if val_m["recall@20_keyterms"] > best[SELECTION_FIELD]:
        best = {SELECTION_FIELD: val_m["recall@20_keyterms"], "epoch": epoch, **val_m}
        os.makedirs(BEST_DIR, exist_ok=True)
        model.save(BEST_DIR)
        with open(os.path.join(RUN_DIR, "best_metrics.json"), "w") as f:
            json.dump({"selection_metric": SELECTION_FIELD, **best, "history": history}, f, indent=2)
        print("✓ Saved BEST model to:", BEST_DIR)

print("BEST:", best)


## Test evaluation (best) + save performance.json


In [ ]:
import os, json
from sentence_transformers import SentenceTransformer
import torch

best_model = SentenceTransformer(BEST_DIR, device=("cuda" if torch.cuda.is_available() else "cpu"))
test_m = eval_metrics(best_model, candidates, test_ex, limit=None)
print(f"TEST Recall@20 keyterms: {test_m['recall@20_keyterms']:.6f} | keywords: {test_m['recall@20_keywords']:.6f}")

perf = {
    "run_name": RUN_NAME,
    "base_embedder": BASE_EMBEDDER,
    "history_turns": HISTORY_TURNS,
    "max_keywords": MAX_KEYWORDS,
    "max_keyterms": MAX_KEYTERMS,
    "selection_metric": SELECTION_FIELD,
    "best_epoch": best.get("epoch"),
    "best_val_recall@20_keyterms": best.get("recall@20_keyterms"),
    "best_val_recall@20_keywords": best.get("recall@20_keywords"),
    "test_recall@20_keyterms": test_m["recall@20_keyterms"],
    "test_recall@20_keywords": test_m["recall@20_keywords"],
    "num_candidates": len(candidates),
    "num_train_examples": len(train_ex),
    "num_val_examples": len(val_ex),
    "num_test_examples": len(test_ex),
}
with open(os.path.join(RUN_DIR, "performance.json"), "w") as f:
    json.dump(perf, f, indent=2)
print("Saved:", os.path.join(RUN_DIR, "performance.json"))


## Build + save shared index with BEST model


In [ ]:
import os, json, numpy as np, faiss

os.makedirs(INDEX_DIR, exist_ok=True)

cand_emb = best_model.encode(candidates, batch_size=256, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
dim = cand_emb.shape[1]
idx = faiss.IndexFlatIP(dim)
idx.add(cand_emb.astype(np.float32))

faiss.write_index(idx, os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "candidates.json"), "w") as f:
    json.dump(candidates, f)

meta = {
    "encoder_dir": BEST_DIR,
    "encoder_base": BASE_EMBEDDER,
    "normalized_embeddings": True,
    "topk": TOPK,
    "candidate_count": len(candidates),
    "best_val_recall@20_keyterms": perf["best_val_recall@20_keyterms"],
    "best_val_recall@20_keywords": perf["best_val_recall@20_keywords"],
    "test_recall@20_keyterms": perf["test_recall@20_keyterms"],
    "test_recall@20_keywords": perf["test_recall@20_keywords"],
}
with open(os.path.join(INDEX_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Saved shared index to:", INDEX_DIR)


## Example predictions (2 test examples)


In [ ]:
import faiss, json, numpy as np, os

idx = faiss.read_index(os.path.join(INDEX_DIR, "index.faiss"))
cands = json.load(open(os.path.join(INDEX_DIR, "candidates.json")))

def uniq(xs):
    seen=set(); out=[]
    for x in xs:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def predict_top_lists(history_text: str, topk=20):
    q = best_model.encode([history_text], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    D, I = idx.search(q, TOPK)
    retrieved = [cands[j] for j in I[0]]

    pred_terms=[]
    pred_words=[]
    for cand in retrieved:
        ct, cw = parse_keyterms_keywords(cand)
        pred_terms.extend(ct)
        pred_words.extend(cw)
    pred_terms = uniq(pred_terms)[:topk]
    pred_words = uniq(pred_words)[:topk]
    return pred_terms, pred_words, retrieved

for ex in test_ex[:2]:
    pt, pw, _ = predict_top_lists(ex["input_text"], topk=20)
    gt_t, gt_w = parse_keyterms_keywords(ex["target_text"])
    ov_t = sorted(set(pt) & set(gt_t))
    ov_w = sorted(set(pw) & set(gt_w))
    print("="*90)
    print("HISTORY:\n", ex["input_text"][:800])
    print("\nGT keyterms:", gt_t[:30])
    print("PRED keyterms@20:", pt)
    print("Overlap keyterms:", ov_t, f"(recall={len(ov_t)}/{len(set(gt_t)) if gt_t else 0})")
    print("\nGT keywords:", gt_w[:30])
    print("PRED keywords@20:", pw)
    print("Overlap keywords:", ov_w, f"(recall={len(ov_w)}/{len(set(gt_w)) if gt_w else 0})")


## Production usage (retrieval)

Artifacts saved to Drive:
- Encoder: `best_model/`
- Index: `shared_index/index.faiss`
- Candidate texts: `shared_index/candidates.json`
- Metrics: `performance.json`, `best_metrics.json`

At inference time you can retrieve candidate strings and then produce **two lists**:
- keyterms@20
- keywords@20
using `parse_keyterms_keywords(...)`.


In [ ]:
# Deepgram runtime injection (example)
# This shows how to pass predicted keywords/keyterms into Deepgram STT.
# Replace DG_API_KEY and audio input as needed.
import os

# Example history (use your live convo history)
example_history = """USER: hi, I need to book a table tonight
SYSTEM: sure, for how many people?
USER: two, around 7pm near downtown"""

# Predict keyterms/keywords from history
pred_keyterms, pred_keywords, _ = predict_top_lists(example_history, topk=20)

# Build Deepgram hint list (merge + de-dupe)
phrases = []
seen = set()
for term in pred_keyterms + pred_keywords:
    t = term.strip()
    if t and t.lower() not in seen:
        seen.add(t.lower())
        phrases.append(t)

print("Predicted keyterms:", pred_keyterms[:10])
print("Predicted keywords:", pred_keywords[:10])
print("Deepgram phrases:", phrases[:20])

# --- Deepgram API call (pseudo, replace with your wiring) ---
# from deepgram import Deepgram
# dg = Deepgram(os.environ["DG_API_KEY"])
# response = dg.transcription.sync_prerecorded(
#     {"buffer": audio_bytes, "mimetype": "audio/wav"},
#     {"model": "nova-2", "smart_format": True, "keywords": phrases}
# )
# print(response)
